<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Create and Configure a Vector Database to Store Document Embeddings**


Estimated time needed: **30** minutes


## Overview


Imagine you are working in a customer support center that receives a high volume of inquiries and tickets every day. Your task is to create a system that can quickly provide support agents with the most relevant information to resolve customer issues. Traditional methods of searching through FAQs or support documents can be slow and inefficient, leading to delayed responses and dissatisfied customers.

To address this challenge, you will use embedding models to convert support documents and past inquiry responses into numerical vectors that capture their semantic content. These vectors will be stored in a vector database, enabling fast and accurate similarity searches. For example, when a support agent receives a new inquiry about a product issue, the system can instantly retrieve similar past inquiries and their resolutions, helping the agent to provide a quicker and more accurate response.


<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/veZYoygp9GqZrIw5f6SD0g/vector%20db.png" width="50%" alt="vector db"/>


In this lab, you will learn how to use vector databases to store embeddings generated from textual data using LangChain. The focus will be on two popular vector databases: Chroma DB and FAISS (Facebook AI Similarity Search). You will also learn how to perform similarity searches in these databases based on a query, enabling efficient retrieval of relevant information. By the end of this lab, you will be able to effectively use vector databases to store and query embeddings, enhancing your data analysis and retrieval capabilities.


## __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-required-libraries">Installing required libraries</a></li>
            <li><a href="#Load-text">Load text</a></li>
            <li><a href="#Split-data">Split data</a></li>
            <li><a href="#Embedding model">Embedding model</a></li>
        </ol>
    </li>
    <li>
        <a href="#Vector-store">Vector store</a>
        <ol>
            <li><a href="#Chroma-DB">Chroma DB</a></li>
            <li><a href="#FIASS-DB">FIASS DB</a></li>
            <li><a href="#Managing-vector-store:-adding,-updating,-and-deleting-entries">Managing vector store: adding, updating, and deleting entries</a></li>
        </ol>
    </li>
</ol>

<a href="#Exercises">Exercises</a>
<ol>
    <li><a href="#Exercise-1---Use-another-query-to-conduct-similarity-search.">Exercise 1. Use another query to conduct similarity search.</a></li>
</ol>


## Objectives

After completing this lab you will be able to:

- Prepare and preprocess documents for embeddings.
- Generate embeddings using watsonx.ai's embedding model.
- Store these embeddings in Chroma DB and FAISS.
- Perform similarity searches to retrieve relevant documents based on new inquiries.


----


## Setup


For this lab, you will use the following libraries:

* [`ibm-watson-ai`](https://ibm.github.io/watsonx-ai-python-sdk/) for using LLMs from IBM's watsonx.ai.
* [`langchain`, `langchain-ibm`, `langchain-community`](https://www.langchain.com/) for using relevant features from Langchain.
* [`chromadb`](https://www.trychroma.com/) is a open-source vector database used to store embeddings.
* [`faiss-cpu`](https://pypi.org/project/faiss-cpu/) is used to support the using of FAISS vector database.


In [1]:
import warnings
warnings.filterwarnings("ignore")

In [15]:
import os
import sys
import subprocess
import importlib.util
import IPython

print("Kernel Python:", sys.executable)

# ------------------------------------------------------------
# 0) Load environment (.env) if python-dotenv is present
# ------------------------------------------------------------
def _load_env_if_possible():
    if importlib.util.find_spec("dotenv") is not None:
        from dotenv import load_dotenv
        load_dotenv(override=True)

_load_env_if_possible()

api_key = os.getenv("OPENAI_API_KEY")
watsonx_api_key = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_url = os.getenv("WATSONX_URL")

if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not set.\n"
        "Create a .env file with:\n"
        "  OPENAI_API_KEY=sk-...\n"
        "and ensure it is loaded before running the notebook."
    )

print("OPENAI_API_KEY loaded ✅ (prefix:", api_key[:6], "…)")
print("WATSON_API_KEY loaded ✅ (prefix:", watsonx_api_key[:6], "…)")
print("WATSONX PROJECT ID loaded ✅", project_id)
print("WATSONX URL loaded ✅", watsonx_url)

Kernel Python: /Users/davidjbrady/venvs/ml_3.11.9_venv/bin/python3.11
OPENAI_API_KEY loaded ✅ (prefix: sk-pro …)
WATSON_API_KEY loaded ✅ (prefix: jS9LCS …)
WATSONX PROJECT ID loaded ✅ 9a16631d-bb24-49ad-ac76-768229034001
WATSONX URL loaded ✅ https://us-south.ml.cloud.ibm.com


### Installing required libraries

In [3]:
# ============================================================
# DEPENDENCY BOOTSTRAP (IBM + LangChain stack, no pinning)
#
# Guarantees:
# - Required packages are installed (any version)
# - Installs only if missing
# - Kernel restarts ONLY if installs occurred
# ============================================================

import sys
import subprocess
import importlib.util
import IPython

print("Kernel Python:", sys.executable)

# import name  -> pip package name
REQUIRED_MODULES = {
    "ibm-watsonx-ai": "ibm-watsonx-ai",
    "langchain": "langchain",
    "langchain-ibm": "langchain-ibm",
    "langchain-community": "langchain-community",
    "chromadb": "chromadb",
    "faiss-cpu": "faiss-cpu",
}

def has_module(mod_name: str) -> bool:
    return importlib.util.find_spec(mod_name) is not None

missing = [
    pip_name
    for mod_name, pip_name in REQUIRED_MODULES.items()
    if not has_module(mod_name)
]

did_install = False

if missing:
    print("Missing packages detected, installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-U", *missing]
    )
    did_install = True
else:
    print("All required packages already installed ✅")

if did_install:
    print("Environment updated. Restarting kernel…")
    IPython.Application.instance().kernel.do_shutdown(True)

print("\n✅ IBM + LangChain dependency bootstrap complete.")

Kernel Python: /Users/davidjbrady/venvs/ml_3.11.9_venv/bin/python3.11
Missing packages detected, installing: ibm-watsonx-ai, langchain-ibm, langchain-community, chromadb, faiss-cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 26.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 20.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 28.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.2 MB/s  0:00:00
  Attempting uninstall: protobuf90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  6/28 [pybase64]s]
    Found existing installation: protobuf 7.34.0━━━━━━━━━━━━━━  6/28 [pybase64]
    Uninstalling protobuf-7.34.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  6/28 [pybase64]
      Successfully uninstalled protobuf-7.34.0━━━━━━━━━━━━━━━━  6/28 [pybase64]
  Attempting uninstall: packaging━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  7/28 [protobuf]
    Found exist

The following steps are prerequisite tasks for conducting this project's topic - vector store. These steps include:

- Loading the source document.
- Splitting the document into chunks.
- Building an embedding model.

The details of these steps have been introduced in previous lessons.


### Load text


A text file has been prepared as the source document for the downstream vector database task.

Now, let's download and load it using LangChain's `TextLoader`.


In [1]:
import requests

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/BYlUHaillwM8EUItaIytHQ/companypolicies.txt"

response = requests.get(url)
response.raise_for_status()  # fail fast if something's wrong

with open("state-of-the-union.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

print("Download complete.")

/Users/davidjbrady/venvs/ml_3.11.9_venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (6.0.0.post1)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


Download complete.


In [2]:
from langchain_community.document_loaders import TextLoader

In [4]:
loader = TextLoader("new-Policies.txt")
data = loader.load()

You can have a look at this document.


In [5]:
data

[Document(metadata={'source': 'new-Policies.txt'}, page_content="1. Code of Conduct\n\nOur Code of Conduct establishes the core values and ethical standards that all members of our organization must adhere to. We are committed to fostering a workplace characterized by integrity, respect, and accountability.\n\nIntegrity: We commit to the highest ethical standards by being honest and transparent in all our dealings, whether with colleagues, clients, or the community. We protect sensitive information and avoid conflicts of interest.\n\nRespect: We value diversity and every individual's contribution. Discrimination, harassment, or any form of disrespect is not tolerated. We promote an inclusive environment where differences are respected, and everyone is treated with dignity.\n\nAccountability: We are responsible for our actions and decisions, complying with all relevant laws and regulations. We aim for continuous improvement and report any breaches of this code, supporting investigations

### Split data


The next step is to split the document using LangChain's text splitter. Here, you will use the `RecursiveCharacterTextSplitter, which is well-suited for this generic text. The following parameters have been set:

- `chunk_size = 100`
- `chunk_overlap = 20`
- `length_function = len`


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
)

In [10]:
chunks = text_splitter.split_documents(data)

Let's take a look at how many chunks you get.


In [11]:
len(chunks)

92

So, in total, you get 215 chunks.


### Embedding model


The following code demonstrates how to build an embedding model using the `watsonx.ai` package.

For this project, the `ibm/slate-125m-english-rtrvr-v2` embedding model will be used.


In [13]:
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames
from langchain_ibm import WatsonxEmbeddings

In [16]:
embed_params = {
    EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 3,
    EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True},
}

watsonx_embedding = WatsonxEmbeddings(
    model_id="ibm/slate-125m-english-rtrvr-v2",
    url=watsonx_url,
    project_id=project_id,
    params=embed_params,
)

The embedding model is formed into the `watsonx_embedding` object.


## Vector store


In this section, you will be guided on how to use two commonly used vector databases: Chroma DB and FAISS DB. You will also see how to perform a similarity search based on an input query using these databases.


### Chroma DB


#### Build the database


First, you need to import `Chroma` from Langchain vector stores.


In [18]:
from langchain_community.vectorstores import Chroma

Next, you need to create an ID list that will be used to assign each chunk a unique identifier, allowing you to track them later in the vector database. The length of this list should match the length of the chunks.

Note: The IDs should be in string format.


In [19]:
ids = [str(i) for i in range(0, len(chunks))]

The next step is to use the embedding model to create embeddings for each chunk and then store them in the Chroma database.

The following code demonstrates how to do this.


In [20]:
vectordb = Chroma.from_documents(chunks, watsonx_embedding, ids=ids)

Now that you have built the vector store named `vectordb`, you can use the method `.collection.get()` to print some of the chunks indexed by their IDs.

Note: Although the chunks are stored in the database in embedding format, when you retrieve and print them by their IDs, the database will return the chunk text information instead of the embedding vectors.


In [21]:
for i in range(3):
    print(vectordb._collection.get(ids=str(i)))

{'ids': ['0'], 'embeddings': None, 'documents': ['1. Code of Conduct'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'new-Policies.txt'}]}
{'ids': ['1'], 'embeddings': None, 'documents': ['Our Code of Conduct establishes the core values and ethical standards that all members of our'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'new-Policies.txt'}]}
{'ids': ['2'], 'embeddings': None, 'documents': ['all members of our organization must adhere to. We are committed to fostering a workplace'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'new-Policies.txt'}]}


You can also use the method `._collection.count()` to see the length of the vector database, which should be the same as the length of chunks.


In [22]:
vectordb._collection.count()

92

#### Similarity search


Similarity search in a vector database involves finding items that are most similar to a given query item based on their vector representations.

In this process, data objects are converted into vectors (which you've already done), and the search algorithm identifies and retrieves those with the closest vector distances to the query, enabling efficient and accurate identification of similar items in large datasets.


LangChain supports similarity search in vector stores using the method `.similarity_search()`.

The following is an example of how to perform a similarity search based on the query "Email policy."

By default, it will return the top four closest vectors to the query.


In [23]:
query = "Email policy"
docs = vectordb.similarity_search(query)
docs

[Document(metadata={'source': 'new-Policies.txt'}, page_content='Accountability: We are responsible for our actions and decisions, complying with all relevant laws'),
 Document(metadata={'source': 'new-Policies.txt'}, page_content='comply with this policy. Regular reviews will ensure it remains relevant with changing technology'),
 Document(metadata={'source': 'new-Policies.txt'}, page_content='personal use is allowed if it does not disrupt work responsibilities.'),
 Document(metadata={'source': 'new-Policies.txt'}, page_content='Equal Opportunity: We are an equal opportunity employer and do not discriminate based on race,')]

You can specify `k = 1` to just retrieve the top one result.


In [24]:
vectordb.similarity_search(query, k = 1)

[Document(metadata={'source': 'new-Policies.txt'}, page_content='Accountability: We are responsible for our actions and decisions, complying with all relevant laws')]

### FIASS DB


FIASS is another vector database that is supported by LangChain.

The process of building and using FAISS is similar to Chroma DB.

However, there may be differences in the retrieval results between FAISS and Chroma DB.


#### Build the database


Build the database and store the embeddings to the database here.


In [25]:
from langchain_community.vectorstores import FAISS

In [26]:
faissdb = FAISS.from_documents(chunks, watsonx_embedding, ids=ids)

Next, print the first three information pieces in the database based on IDs.


In [27]:
for i in range(3):
    print(faissdb.docstore.search(str(i)))

page_content='1. Code of Conduct' metadata={'source': 'new-Policies.txt'}
page_content='Our Code of Conduct establishes the core values and ethical standards that all members of our' metadata={'source': 'new-Policies.txt'}
page_content='all members of our organization must adhere to. We are committed to fostering a workplace' metadata={'source': 'new-Policies.txt'}


#### Similarity search


Let's do a similarity search again using FIASS DB on the same query.


In [28]:
query = "Email policy"
docs = faissdb.similarity_search(query)
docs

[Document(id='10', metadata={'source': 'new-Policies.txt'}, page_content='Accountability: We are responsible for our actions and decisions, complying with all relevant laws'),
 Document(id='69', metadata={'source': 'new-Policies.txt'}, page_content='comply with this policy. Regular reviews will ensure it remains relevant with changing technology'),
 Document(id='75', metadata={'source': 'new-Policies.txt'}, page_content='personal use is allowed if it does not disrupt work responsibilities.'),
 Document(id='24', metadata={'source': 'new-Policies.txt'}, page_content='Equal Opportunity: We are an equal opportunity employer and do not discriminate based on race,')]

The retrieve results based on the similarity search seem to be the same as with the Chroma DB.

You can try with other queries or documents to see if they follow the same situation.


### Managing vector store: Adding, updating, and deleting entries


There might be situations where new documents come into your RAG application that you want to add to the current vector database, or you might need to delete some existing documents from the database. Additionally, there may be updates to some of the documents in the database that require updating.

The following sections will guide you on how to perform these tasks. You will use the Chroma DB as an example.


#### Add


Imagine you have a new piece of text information that you want to add to the vector database. First, this information should be formatted into a document object.


In [29]:
text = "Instructlab is the best open source tool for fine-tuning a LLM."

In [30]:
from langchain_core.documents import Document

Form the text into a `Document` object named `new_chunk`.


In [31]:
new_chunk =  Document(
    page_content=text,
    metadata={
        "source": "ibm.com",
        "page": 1
    }
)

Then, the new chunk should be put into a list as the vector database only accepts documents in a list.


In [32]:
new_chunks = [new_chunk]

Before you add the document to the vector database, since there are 215 chunks with IDs from 0 to 214, if you print ID 215, the document should show no values. Let's validate it.


In [33]:
print(vectordb._collection.get(ids=['215']))

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


Next, you can use the method `.add_documents()` to add this `new_chunk`. In this method, you should assign an ID to the document. Since there are already IDs from 0 to 214, you can assign ID 215 to this document. The ID should be in string format and placed in a list.


In [34]:
vectordb.add_documents(
    new_chunks,
    ids=["215"]
)

['215']

Now you can count the length of the vector database again to see if it has increased by one.


In [35]:
vectordb._collection.count()

93

You can then print this newly added document from the database by its ID.


In [36]:
print(vectordb._collection.get(ids=['215']))

{'ids': ['215'], 'embeddings': None, 'documents': ['Instructlab is the best open source tool for fine-tuning a LLM.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'page': 1, 'source': 'ibm.com'}]}


#### Update


Imagine you want to update the content of a document that is already stored in the database. The following code demonstrates how to do this.


Still, you need to form the updated text into a `Document` object.


In [37]:
update_chunk =  Document(
    page_content="Instructlab is a perfect open source tool for fine-tuning a LLM.",
    metadata={
        "source": "ibm.com",
        "page": 1
    }
)

Then, you can use the method `.update_document()` to update the specific stored information indexing by its ID.


In [38]:
vectordb.update_document(
    '215',
    update_chunk,
)

In [39]:
print(vectordb._collection.get(ids=['215']))

{'ids': ['215'], 'embeddings': None, 'documents': ['Instructlab is a perfect open source tool for fine-tuning a LLM.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'page': 1, 'source': 'ibm.com'}]}


As you can see, the document information has been updated.


#### Delete


If you want to delete documents from the vector database, you can use the method `_collection.delete()` and specify the document ID to delete it.


In [40]:
vectordb._collection.delete(ids=['215'])

In [41]:
print(vectordb._collection.get(ids=['215']))

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


As you can see, now that document is empty.


# Exercises


### Exercise 1 - Use another query to conduct similarity search.

Can you use another query to conduct the similarity search?


In [ ]:
# Your code here

<details>
    <summary>Click here for solution</summary>

```python
query = "Smoking policy"
docs = vectordb.similarity_search(query)
docs
```

</details>


## Authors


[Kang Wang](https://author.skills.network/instructors/kang_wang)

Kang Wang is a Data Scientist in IBM. He is also a PhD Candidate in the University of Waterloo.


### Other Contributors


[Joseph Santarcangelo](https://author.skills.network/instructors/joseph_santarcangelo)

Joseph has a Ph.D. in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.


```{## Change Log}
```


```{|Date (YYYY-MM-DD)|Version|Changed By|Change Description||-|-|-|-||2024-07-24|0.1|Kang Wang|Create the lab|}
```



Copyright © IBM Corporation. All rights reserved.
